https://developers.binance.info/docs/zh-CN/binance-spot-api-docs/web-socket-streams

按Symbol的最优挂单信息

实时推送指定交易对最优挂单信息 多个 <symbol>@bookTicker 可以订阅在一个WebSocket连接上

Stream 名称: <symbol>@bookTicker

更新速度: 实时

Payload:

{
  "u":400900217,     // order book updateId
  "s":"BNBUSDT",     // 交易对
  "b":"25.35190000", // 买单最优挂单价格
  "B":"31.21000000", // 买单最优挂单数量
  "a":"25.36520000", // 卖单最优挂单价格
  "A":"40.66000000"  // 卖单最优挂单数量
}

In [1]:
import websocket
import threading
import json

latest_bbos = {}
allowed_symbols = ["BTCUSDT", "ETHUSDT", "BNBUSDT", "SOLUSDT", "XRPUSDT", "DOGEUSDT", "ADAUSDT"]

def on_message(ws, message):
    json_message = json.loads(message)
    print("Received new orderbook L1 data")
    symbol = json_message['s'].lower()
    bbo = {
        'b': float(json_message['b']),
        'B': float(json_message['B']),
        'a': float(json_message['a']),
        'A': float(json_message['A'])
    }
    latest_bbos[symbol] = bbo
    print(f"{symbol.upper()} BBO: {bbo}")

def on_error(ws, error):
    print(f"Error: {error}")

def on_close(ws):
    print("### Closed Connection ###")

def on_open(ws):
    """
    ws连接刚建立时不稳定，发送订阅请求由新建线程完成，而主线程继续处理连接维护
    """
    def run(*args):
        subscribe_message = {
            "method": "SUBSCRIBE",
            "params": [f'{symbol.lower()}@bookTicker' for symbol in allowed_symbols],
            "id": 1
            }
        
        ws.send(json.dumps(subscribe_message))
    
    threading.Thread(target=run).start()      # 创建新线程，不阻塞主线程

In [3]:
socket_url = "wss://stream.binance.com:9443/ws"

ws = websocket.WebSocketApp(socket_url,
                            on_open=on_open,
                            on_message=on_message,
                            on_error=on_error,
                            on_close=on_close)

wst = threading.Thread(target=ws.run_forever)
wst.daemon = True
wst.start()

try: 
    while True:
        pass
    
except KeyboardInterrupt:
    ws.close()
    

Received new orderbook L1 data
Error: 's'
Received new orderbook L1 data
BNBUSDT BBO: {'b': 900.39, 'B': 28.756, 'a': 900.4, 'A': 12.313}
Received new orderbook L1 data
ETHUSDT BBO: {'b': 3098.37, 'B': 102.0633, 'a': 3098.38, 'A': 6.7487}
Received new orderbook L1 data
ETHUSDT BBO: {'b': 3098.37, 'B': 102.0633, 'a': 3098.38, 'A': 6.9487}
Received new orderbook L1 data
ETHUSDT BBO: {'b': 3098.37, 'B': 102.0633, 'a': 3098.38, 'A': 6.7487}
Received new orderbook L1 data
SOLUSDT BBO: {'b': 136.68, 'B': 582.519, 'a': 136.69, 'A': 92.795}
Received new orderbook L1 data
ETHUSDT BBO: {'b': 3098.37, 'B': 102.2633, 'a': 3098.38, 'A': 6.7487}
Received new orderbook L1 data
ETHUSDT BBO: {'b': 3098.37, 'B': 96.0444, 'a': 3098.38, 'A': 6.7487}
Received new orderbook L1 data
ETHUSDT BBO: {'b': 3098.37, 'B': 95.8444, 'a': 3098.38, 'A': 6.7487}
Received new orderbook L1 data
ETHUSDT BBO: {'b': 3098.37, 'B': 96.0444, 'a': 3098.38, 'A': 6.7487}
Received new orderbook L1 data
BTCUSDT BBO: {'b': 90729.66, 